# 3 · Propagating probability through one graph

Exact, enumerated, sampled and naive propagation, and what each partial view sees. See
[Lesson 3](../docs/lessons/03-propagating-probability-and-likelihood.md).

In [1]:
import matplotlib.pyplot as plt

from cpsrbd import plotting

plotting.use_style()

In [2]:
from cpsrbd.consequence_driven_analysis import navigation_case_study
from cpsrbd.probability_likelihood_propagation import (
    bottom_up_probability,
    enumerate_probability,
    exact_probability,
    monte_carlo_probability,
    origin_decomposition,
)

model = navigation_case_study()
print(f"exact        {exact_probability(model):.6e}")
print(f"enumeration  {enumerate_probability(model):.6e}")
print(f"naive        {bottom_up_probability(model):.6e}")
mc = monte_carlo_probability(model, samples=500_000, seed=7)
print(f"Monte Carlo  {mc.estimate:.3e}  95 % CI [{mc.low:.3e}, {mc.high:.3e}]")

exact        1.015139e-03


enumeration  1.015139e-03
naive        1.016202e-03
Monte Carlo  9.540e-04  95 % CI [8.722e-04, 1.043e-03]


Where shared events sit under an AND, the tree formulas fail badly:

In [3]:
from cpsrbd.rbd import bridge, to_unified

for q in (0.01, 0.1, 0.3):
    m = to_unified(bridge([1 - q] * 5))
    exact, naive = exact_probability(m), bottom_up_probability(m)
    print(f"q = {q:4}: exact {exact:.4e}  naive {naive:.4e}  error {(naive - exact) / exact:+.0%}")

q = 0.01: exact 2.0195e-04  naive 3.4934e-07  error -100%
q =  0.1: exact 2.1520e-02  naive 2.6512e-03  error -88%
q =  0.3: exact 1.9836e-01  naive 1.1227e-01  error -43%


Each team's view, and the interaction gap between them:

In [4]:
d = origin_decomposition(model)
print(f"physical only  {d.physical:.3e}")
print(f"cyber only     {d.cyber:.3e}")
print(f"combined       {d.separate:.3e}")
print(f"unified        {d.full:.3e}")
print(f"gap            {d.gap:.3e}  ({d.gap_share:.0%} of the risk)")

physical only  5.047e-04
cyber only     1.945e-04
combined       6.991e-04
unified        1.015e-03
gap            3.160e-04  (31% of the risk)
